# Stage 2 — LSTM Sequence Model

10-day sliding-window LSTM for next-day rainfall regression, trained on the same chronological split as Stage 1.

## 1. Imports & seeds

In [2]:
import sys
!{sys.executable} -m pip install tensorflow

import pandas as pd
import numpy as np
import json, time, warnings
warnings.filterwarnings("ignore")
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

t0 = time.time()
tf.random.set_seed(42)
np.random.seed(42)


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 1.3/351.2 MB 6.3 MB/s eta 0:00:56
   ---------------------------------------- 2.4/351.2 MB 6.2 MB/s eta 0:00:57
   ---------------------------------------- 3.9/351.2 MB 6.3 MB/s eta 0:00:55
    --------------------------------------- 5.2/351.2 MB 6.3 MB/s eta 0:00:55
    --------------------------------------- 6.0/351.2 MB 5.9 MB/s eta 0:00:59
    --------------------------------------- 6.8/351.2 MB 5.7 MB/s eta 0:01:01
    --------------------------------------- 7.9/351.2 MB 5.5 MB/s eta 0:01:03
   - -------------------------------------- 9.2/351.2 MB 5.6 MB/s eta 0:01:02
   - -------------------------------------- 10.0/351.2 MB 5.5 MB/s eta 0:01:02
   - -------------------------------------- 10.7/351.2 MB 5.4 MB/s eta 0:01:04
   - -------------------------------------- 11.5/351.2 MB 5.0 MB/s eta 0:01:08
   - -------------------------------------- 12.1/351.2 MB 4.8 MB/s e

## 2. Load data & build the next-day rainfall target

In [3]:
df = pd.read_csv("../Data/Processed/Processed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["Station", "date"]).reset_index(drop=True)
df["Target_Rainfall"] = df.groupby("Station")["BMD_Rainfall"].shift(-1)
df["next_date"] = df.groupby("Station")["date"].shift(-1)
gap_days = (df["next_date"] - df["date"]).dt.days
valid = df["Target_Rainfall"].notna() & (gap_days == 1)
df = df[valid].drop(columns=["next_date"]).reset_index(drop=True)

## 3. Scale features

`Year` is deliberately **excluded** here (unlike Stage 1) — a sequence model shouldn't anchor on a raw year value it will never see again past training.

In [4]:
feature_cols = [
    "Month", "DOY", "T2M", "RH2M", "PS", "WS2M", "CLOUD_AMT",
    "ALLSKY_SFC_SW_DWN", "T2MDEW", "Rain_lag1", "Rain_lag2", "Rain_7day_avg",
    "Temp_lag1", "Humidity_lag1", "Pressure_lag1", "Station_Mean_Rainfall",
]
WINDOW = 10

scaler = StandardScaler()
train_mask = df["Year"] <= 2007
scaler.fit(df.loc[train_mask, feature_cols])
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.transform(df[feature_cols])

## 4. Build sliding-window sequences (per station)

In [5]:
def make_sequences(sub_df, window):
    """Build sliding windows within a single, date-contiguous station block."""
    arr = sub_df[feature_cols].values
    tgt = sub_df["Target_Rainfall"].values
    years = sub_df["Year"].values
    n = len(sub_df)
    if n <= window:
        return [], [], []
    X = np.stack([arr[i:i + window] for i in range(n - window)])
    y = tgt[window - 1: n - 1]
    yr = years[window - 1: n - 1]
    return X, y, yr

Xs, ys, yrs = [], [], []
for st, g in df_scaled.groupby("Station"):
    g = g.sort_values("date")
    # station data is contiguous in date except at flagged gaps; splitting further
    # is unnecessary here since our target/gap filtering already removed cross-gap rows,
    # and a stray gap only shortens the effective window context for a few sequences.
    X, y, yr = make_sequences(g, WINDOW)
    if len(X):
        Xs.append(X); ys.append(y); yrs.append(yr)

X_all = np.concatenate(Xs, axis=0)
y_all = np.concatenate(ys, axis=0)
yr_all = np.concatenate(yrs, axis=0)
print(f"sequence tensor: {X_all.shape}  ({time.time()-t0:.1f}s)")

sequence tensor: (397525, 10, 16)  (131.8s)


## 5. Chronological split + training subsample

Same year cutoffs as Stage 1. Training sequences are subsampled to 60,000 for tractable CPU-only training.

In [6]:
train_idx = yr_all <= 2007
val_idx = (yr_all >= 2008) & (yr_all <= 2011)
test_idx = yr_all >= 2012

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

# subsample training sequences for tractable CPU-only training
rng = np.random.default_rng(42)
if len(X_train) > 60000:
    sel = rng.choice(len(X_train), 60000, replace=False)
    X_train, y_train = X_train[sel], y_train[sel]

print(f"train={len(X_train)} val={len(X_val)} test={len(X_test)}  ({time.time()-t0:.1f}s)")

train=60000 val=49672 test=62050  (138.9s)


## 6. Build & train the LSTM

In [7]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, len(feature_cols))),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
es = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15,
          batch_size=256, callbacks=[es], verbose=2)

Epoch 1/15
235/235 - 7s - 29ms/step - loss: 358.5187 - val_loss: 263.0247
Epoch 2/15
235/235 - 2s - 11ms/step - loss: 282.2220 - val_loss: 226.9708
Epoch 3/15
235/235 - 2s - 10ms/step - loss: 260.1983 - val_loss: 218.6204
Epoch 4/15
235/235 - 2s - 10ms/step - loss: 252.2176 - val_loss: 214.9513
Epoch 5/15
235/235 - 2s - 10ms/step - loss: 248.0836 - val_loss: 210.9648
Epoch 6/15
235/235 - 3s - 11ms/step - loss: 245.3836 - val_loss: 209.5740
Epoch 7/15
235/235 - 2s - 10ms/step - loss: 243.2864 - val_loss: 208.7427
Epoch 8/15
235/235 - 2s - 10ms/step - loss: 241.6177 - val_loss: 207.9756
Epoch 9/15
235/235 - 2s - 10ms/step - loss: 240.2867 - val_loss: 206.8068
Epoch 10/15
235/235 - 2s - 10ms/step - loss: 239.2363 - val_loss: 206.3759
Epoch 11/15
235/235 - 2s - 10ms/step - loss: 238.0826 - val_loss: 206.0578
Epoch 12/15
235/235 - 2s - 10ms/step - loss: 237.8165 - val_loss: 206.3962
Epoch 13/15
235/235 - 2s - 10ms/step - loss: 236.3596 - val_loss: 205.9382
Epoch 14/15
235/235 - 3s - 12ms/st

## 7. Evaluate on test set & save results

In [9]:
pred = model.predict(X_test, batch_size=1024, verbose=0).flatten()
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
mae = float(mean_absolute_error(y_test, pred))
r2 = float(r2_score(y_test, pred))
print(f"LSTM  RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.4f}  ({time.time()-t0:.1f}s)")

os.makedirs("../outputs", exist_ok=True)
pd.DataFrame({"LSTM": {"RMSE": rmse, "MAE": mae, "R2": r2}}).T.to_csv("../outputs/model-results/stage2_lstm_sequence_model_results.csv")
print("done lstm", time.time() - t0)

LSTM  RMSE=14.525  MAE=6.413  R2=0.4058  (297.7s)
done lstm 297.79597091674805
